<a href="https://colab.research.google.com/github/nekoncu/ml-olympiad-red-task-BerdnikovArtyom/blob/main/alignment_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Выравнивание языковой модели: SFT → DPO → Reward Model → DPO → SimPO

Полный пайплайн выравнивания **Qwen/Qwen3-4B-Instruct-2507** (QLoRA 4-bit, Colab/Kaggle T4).

Пайплайн:
1. **SFT** на `kid_adult` → перенос простого стиля → метрика `P_simple` на `public_test_style`
2. **DPO (стиль)** на `kid_adult` поверх SFT → закрепление предпочтения → `P_simple`
3. **Reward Model** на `good_bad` → pairwise accuracy на `public_test_quality`
4. **DPO (качество)** на `good_bad` → implicit-preference accuracy (length-normalized logprob)
5. **SimPO** на `good_bad` → та же метрика, сравнение с DPO

Всюду: `seed = 42`, генерация greedy (`do_sample=False`).

In [1]:
# === Установка зависимостей ===
# Версии зафиксированы для воспроизводимости
!pip install -q -U "transformers==4.55.0" "trl==0.19.1" "peft==0.16.0" \
    "bitsandbytes==0.46.0" "datasets==3.6.0" "accelerate==1.8.1" \
    "scikit-learn==1.7.2" "scipy"


In [2]:
# === Данные ===
# Вариант А (Colab): загрузите архив ml-olympiad-red-task.zip в сессию и распакуйте.
# Вариант Б (Kaggle): подключите датасет и поправьте пути ниже.
import os
if not os.path.exists("ml-olympiad-red-task"):
    !unzip -q -o ml-olympiad-red-task*.zip -d .

DATA_DIR    = "ml-olympiad-red-task/data"
METRICS_DIR = "ml-olympiad-red-task/metrics"
assert os.path.exists(f"{DATA_DIR}/kid_adult.jsonl"), "Проверь путь к данным!"


In [3]:
# === Импорты, seed, константы ===
import json, gc, random
import numpy as np
import torch
from scipy import sparse

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
DEVICE = "cuda"
print(torch.cuda.get_device_name(0))


Tesla T4


In [4]:
# === Загрузка данных ===
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

kid_adult    = read_jsonl(f"{DATA_DIR}/kid_adult.jsonl")     # prompt / kid / adult
good_bad     = read_jsonl(f"{DATA_DIR}/good_bad.jsonl")      # instruction / chosen / rejected
test_style   = read_jsonl(f"{DATA_DIR}/public_test_style.jsonl")
test_quality = read_jsonl(f"{DATA_DIR}/public_test_quality.jsonl")

print(len(kid_adult), len(good_bad), len(test_style), len(test_quality))


1489 2226 50 50


In [5]:
# === Стилевой классификатор: P_simple ===
# style_clf.pkl = {'clf': LogisticRegression, 'vecs': (tfidf_word, tfidf_char)}
# Класс 1 = простой стиль (проверено на kid/adult из public_test_style)
import pickle

with open(f"{METRICS_DIR}/style_clf.pkl", "rb") as f:
    _style = pickle.load(f)
_style_clf, (_v_word, _v_char) = _style["clf"], _style["vecs"]

def p_simple(texts):
    X = sparse.hstack([_v_word.transform(texts), _v_char.transform(texts)])
    return _style_clf.predict_proba(X)[:, 1]

# sanity-check: kid должен быть ~1, adult ~0
print("kid  :", p_simple([r["kid"] for r in test_style]).mean().round(3))
print("adult:", p_simple([r["adult"] for r in test_style]).mean().round(3))


kid  : 0.975
adult: 0.018


In [6]:
# === Токенизатор и утилиты загрузки модели ===
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4 — fp16
)

def load_base():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=BNB, device_map="auto",
        torch_dtype=torch.float16, attn_implementation="eager",
    )
    m.config.use_cache = False
    return m

def free(*objs):
    for o in objs:
        del o
    gc.collect(); torch.cuda.empty_cache()

def chat_prompt(user_text):
    """Промпт в формате чата Qwen3 с генерационным префиксом ассистента."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False, add_generation_prompt=True,
    )


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [7]:
# === Единая функция генерации (greedy, seed=42) и метрики P_simple ===
from transformers import set_seed

@torch.no_grad()
def generate_answers(model, prompts, max_new_tokens=200, batch_size=8):
    set_seed(SEED)
    model.eval()
    tokenizer.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), batch_size):
        batch = [chat_prompt(p) for p in prompts[i:i+batch_size]]
        enc = tokenizer(batch, return_tensors="pt", padding=True).to(DEVICE)
        gen = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,            # greedy — детерминированно
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.pad_token_id,
        )
        for j in range(len(batch)):
            new = gen[j][enc["input_ids"].shape[1]:]
            outs.append(tokenizer.decode(new, skip_special_tokens=True).strip())
    tokenizer.padding_side = "right"
    return outs

def eval_p_simple(model, label=""):
    prompts = [r["prompt"] for r in test_style]
    answers = generate_answers(model, prompts)
    score = float(np.mean(p_simple(answers)))
    print(f"[{label}] средний P_simple на public_test_style = {score:.4f}")
    return score, answers


## Задача 0 (не обязательна, но полезна) — базовая модель до обучения

Точка отсчёта: базовая модель отвечает в «сложном регистре», ожидаем низкий `P_simple`.

In [8]:
base = load_base()
base_score, base_answers = eval_p_simple(base, "BASE")
print(base_answers[0][:300])
free(base)


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

[BASE] средний P_simple на public_test_style = 0.3209
Правильный ответ: **Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.**

Это объяснение корректно и полно. Продавцы используют скидки и распродажи для нескольких ключевых целей:

1. **Ускорения продаж старых товаров** — товары, которые уже 


## Задача 1 — SFT: перенос стиля

Обучаем на парах `prompt → kid` (простой ответ). Датасет в формате **prompt/completion** —
TRL считает лосс только по completion (стиль ответа), а не по промпту.

Гиперпараметры (подобраны под ~1.5k примеров, T4, QLoRA):
LoRA r=16, α=32, dropout=0.05, все linear-проекции; lr=2e-4 (cosine), 2 эпохи, эфф. батч 16.

In [9]:
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

LORA = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

EOS = tokenizer.eos_token  # "<|im_end|>"

sft_ds = Dataset.from_list([
    {"prompt": chat_prompt(r["prompt"]), "completion": r["kid"] + EOS}
    for r in kid_adult
])
print(sft_ds[0]["prompt"][-120:], "||", sft_ds[0]["completion"][:120])


ез стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.<|im_end|>
<|im_start|>assistant
 || Камера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и со


In [10]:
seed_everything()
model = load_base()
model = prepare_model_for_kbit_training(model)

sft_args = SFTConfig(
    output_dir="out_sft",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    max_length=1024,
    seed=SEED,
    report_to="none",
    gradient_checkpointing=True,
    completion_only_loss=True,   # лосс только по ответу
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_ds,
    peft_config=LORA,
    processing_class=tokenizer,
)
trainer.train()
trainer.model.save_pretrained("adapters/sft")
sft_model = trainer.model
free(trainer)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Step,Training Loss
20,1.794900
40,1.340600
60,1.278600
80,1.243500
100,1.165500
120,0.965400
140,0.964400
160,0.969100
180,0.968900


In [11]:
# === МЕТРИКА ЗАДАЧИ 1 ===
task1_score, task1_answers = eval_p_simple(sft_model, "TASK 1 / SFT")
print("Пример ответа:", task1_answers[0][:300])


[TASK 1 / SFT] средний P_simple на public_test_style = 0.9704
Пример ответа: Продавцы делают скидки, чтобы старые вещи быстро ушли из магазина. Это освободит место для новых, красивых и модных игрушек. А еще так они приглашают друзей и соседей: «Смотри, тут всё дешевле!»


**Ответ на задачу 1** — интервал, в который попало напечатанное выше значение `P_simple`:
А) < 0.4 Б) 0.4–0.7 В) 0.7–0.9 Г) 0.9–1.0

---

## Задача 2 — DPO по стилю

Поверх SFT-адаптера прогоняем DPO на тех же данных: `chosen = kid`, `rejected = adult`.
Reference-модель TRL создаёт автоматически (тот же PEFT-модуль с отключённым адаптером).
lr на порядки меньше SFT (5e-6), β=0.1, 1 эпоха.

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_style_ds = Dataset.from_list([
    {"prompt": chat_prompt(r["prompt"]),
     "chosen": r["kid"] + EOS,
     "rejected": r["adult"] + EOS}
    for r in kid_adult
])

seed_everything()
dpo_args = DPOConfig(
    output_dir="out_dpo_style",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    beta=0.1,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    max_length=1024,
    max_prompt_length=512,
    seed=SEED,
    report_to="none",
    gradient_checkpointing=True,
)

dpo_trainer = DPOTrainer(
    model=sft_model,          # продолжаем обучать SFT-адаптер
    ref_model=None,           # ref = адаптер отключён
    args=dpo_args,
    train_dataset=dpo_style_ds,
    processing_class=tokenizer,
)
dpo_trainer.train()
dpo_trainer.model.save_pretrained("adapters/dpo_style")
dpo_style_model = dpo_trainer.model
free(dpo_trainer)


In [ ]:
# === МЕТРИКА ЗАДАЧИ 2 ===
task2_score, task2_answers = eval_p_simple(dpo_style_model, "TASK 2 / DPO-style")
print(f"Рост относительно SFT: {task1_score:.4f} -> {task2_score:.4f}")


**Ответ на задачу 2** — интервал `P_simple` (те же А/Б/В/Г).

---

## Задача 3 — Reward Model

Обучаем оценщика качества на `good_bad`: скаляр-оценка, лосс `-log σ(r_chosen − r_rejected)`.
Берём ту же базу как `AutoModelForSequenceClassification(num_labels=1)` + QLoRA (SEQ_CLS,
голова `score` обучается полностью). Важно: обе стороны пары в **простом** стиле — оценщик
вынужден учиться на сути, а не на тоне.

Текст для оценки = промпт + ответ в чат-формате (тот же формат для chosen и rejected).

In [ ]:
# Освобождаем память перед RM
free(sft_model, dpo_style_model)
torch.cuda.empty_cache()

from transformers import AutoModelForSequenceClassification
from trl import RewardTrainer, RewardConfig

def rm_text(instruction, answer):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction},
         {"role": "assistant", "content": answer}],
        tokenize=False,
    )

rm_ds = Dataset.from_list([
    {"chosen":  rm_text(r["instruction"], r["chosen"]),
     "rejected": rm_text(r["instruction"], r["rejected"])}
    for r in good_bad
])

seed_everything()
rm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=1,
    quantization_config=BNB, device_map="auto", torch_dtype=torch.float16,
)
rm_model.config.pad_token_id = tokenizer.pad_token_id
rm_model.config.use_cache = False
rm_model = prepare_model_for_kbit_training(rm_model)

RM_LORA = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    modules_to_save=["score"],
)

rm_args = RewardConfig(
    output_dir="out_rm",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    max_length=1024,
    seed=SEED,
    report_to="none",
    gradient_checkpointing=True,
)

rm_trainer = RewardTrainer(
    model=rm_model,
    args=rm_args,
    train_dataset=rm_ds,
    peft_config=RM_LORA,
    processing_class=tokenizer,
)
rm_trainer.train()
rm_trainer.model.save_pretrained("adapters/reward_model")
rm_model = rm_trainer.model
free(rm_trainer)


In [ ]:
# === МЕТРИКА ЗАДАЧИ 3: pairwise accuracy на public_test_quality ===
@torch.no_grad()
def rm_score(model, texts, batch_size=4):
    model.eval()
    scores = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i+batch_size], return_tensors="pt",
                        padding=True, truncation=True, max_length=1024).to(DEVICE)
        out = model(**enc)
        scores.extend(out.logits.squeeze(-1).float().cpu().tolist())
    return scores

chosen_texts   = [rm_text(r["prompt"], r["chosen"])   for r in test_quality]
rejected_texts = [rm_text(r["prompt"], r["rejected"]) for r in test_quality]

s_c = rm_score(rm_model, chosen_texts)
s_r = rm_score(rm_model, rejected_texts)
task3_acc = float(np.mean([c > r for c, r in zip(s_c, s_r)]))
print(f"[TASK 3 / RM] pairwise accuracy на public_test_quality = {task3_acc:.4f}")


**Ответ на задачу 3** — интервал accuracy:
А) < 0.6 Б) 0.6–0.75 В) 0.75–0.9 Г) 0.9–1.0

---

## Задача 4 — DPO по качеству

Стиль закреплён (задача 2), теперь оптимизируем содержание: DPO на `good_bad`
поверх модели из задачи 2.

**Метрика — implicit-preference accuracy:** доля пар, где модель даёт хорошему ответу
больший **средний logprob на токен** (сумма logprob токенов ответа / число токенов).
Без генерации, детерминированно. Нормировка на длину обязательна.

In [ ]:
# Освобождаем RM
free(rm_model)
torch.cuda.empty_cache()

from peft import PeftModel

# Возвращаем модель задачи 2: база + адаптер dpo_style
seed_everything()
base = load_base()
base = prepare_model_for_kbit_training(base)
model_q = PeftModel.from_pretrained(base, "adapters/dpo_style", is_trainable=True)

dpo_quality_ds = Dataset.from_list([
    {"prompt": chat_prompt(r["instruction"]),
     "chosen": r["chosen"] + EOS,
     "rejected": r["rejected"] + EOS}
    for r in good_bad
])

dpo_q_args = DPOConfig(
    output_dir="out_dpo_quality",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    beta=0.1,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    max_length=1024,
    max_prompt_length=512,
    seed=SEED,
    report_to="none",
    gradient_checkpointing=True,
)

dpo_q_trainer = DPOTrainer(
    model=model_q,
    ref_model=None,
    args=dpo_q_args,
    train_dataset=dpo_quality_ds,
    processing_class=tokenizer,
)
dpo_q_trainer.train()
dpo_q_trainer.model.save_pretrained("adapters/dpo_quality")
dpo_quality_model = dpo_q_trainer.model
free(dpo_q_trainer)


In [ ]:
# === Implicit-preference accuracy (length-normalized mean logprob) ===
@torch.no_grad()
def avg_logprob(model, user_text, answer):
    """Средний logprob на токен ответа при teacher forcing."""
    prompt = chat_prompt(user_text)
    p_ids = tokenizer(prompt, return_tensors="pt").input_ids
    full  = tokenizer(prompt + answer + EOS, return_tensors="pt").input_ids
    full = full.to(DEVICE)
    n_prompt = p_ids.shape[1]

    logits = model(full).logits[0].float()          # [T, V]
    logprobs = torch.log_softmax(logits, dim=-1)
    # предсказание токена t делается на позиции t-1
    tgt = full[0, n_prompt:]                        # токены ответа (+eos)
    lp  = logprobs[n_prompt-1:-1, :].gather(1, tgt.unsqueeze(1)).squeeze(1)
    return (lp.sum() / lp.numel()).item()           # сумма / число токенов

def implicit_pref_accuracy(model, label=""):
    model.eval()
    wins = []
    for r in test_quality:
        lc = avg_logprob(model, r["prompt"], r["chosen"])
        lr = avg_logprob(model, r["prompt"], r["rejected"])
        wins.append(lc > lr)
    acc = float(np.mean(wins))
    print(f"[{label}] implicit-preference accuracy = {acc:.4f}")
    return acc


In [ ]:
# === МЕТРИКА ЗАДАЧИ 4 ===
task4_acc = implicit_pref_accuracy(dpo_quality_model, "TASK 4 / DPO-quality")


**Ответ на задачу 4** — интервал accuracy (А/Б/В/Г как в задаче 3).

---

## Задача 5 — SimPO

Та же цель, что в задаче 4, но без reference-модели: в SimPO «наградой» служит сам
средний logprob на токен (length-normalized) с целевым отступом γ. В TRL это
`CPOTrainer` с `loss_type="simpo"` и `cpo_alpha=0`.

Стартуем из той же точки, что и задача 4 (модель после задачи 2) — честное сравнение
алгоритмов на одном датасете. Типичные гиперпараметры SimPO: β≈2.0, γ≈1.0.

In [ ]:
free(dpo_quality_model)
torch.cuda.empty_cache()

from trl import CPOTrainer, CPOConfig

seed_everything()
base = load_base()
base = prepare_model_for_kbit_training(base)
model_s = PeftModel.from_pretrained(base, "adapters/dpo_style", is_trainable=True)

simpo_args = CPOConfig(
    output_dir="out_simpo",
    loss_type="simpo",
    cpo_alpha=0.0,            # чистый SimPO, без NLL-члена
    beta=2.0,
    simpo_gamma=1.0,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    fp16=True,
    max_length=1024,
    max_prompt_length=512,
    seed=SEED,
    report_to="none",
    gradient_checkpointing=True,
)

simpo_trainer = CPOTrainer(
    model=model_s,
    args=simpo_args,
    train_dataset=dpo_quality_ds,   # тот же датасет, что в задаче 4
    processing_class=tokenizer,
)
simpo_trainer.train()
simpo_trainer.model.save_pretrained("adapters/simpo")
simpo_model = simpo_trainer.model
free(simpo_trainer)


In [ ]:
# === МЕТРИКА ЗАДАЧИ 5 ===
task5_acc = implicit_pref_accuracy(simpo_model, "TASK 5 / SimPO")
print(f"Сравнение: DPO = {task4_acc:.4f} vs SimPO = {task5_acc:.4f}")


## Итоговая сводка

In [ ]:
print("="*60)
print(f"Задача 1  SFT        P_simple            = {task1_score:.4f}")
print(f"Задача 2  DPO-style  P_simple            = {task2_score:.4f}")
print(f"Задача 3  RM         pairwise accuracy   = {task3_acc:.4f}")
print(f"Задача 4  DPO-qual   implicit-pref acc   = {task4_acc:.4f}")
print(f"Задача 5  SimPO      implicit-pref acc   = {task5_acc:.4f}")
print("="*60)

def interval_style(x):
    return "А (<0.4)" if x < 0.4 else "Б (0.4–0.7)" if x < 0.7 else "В (0.7–0.9)" if x < 0.9 else "Г (0.9–1.0)"
def interval_acc(x):
    return "А (<0.6)" if x < 0.6 else "Б (0.6–0.75)" if x < 0.75 else "В (0.75–0.9)" if x < 0.9 else "Г (0.9–1.0)"

print("Задача 1:", interval_style(task1_score))
print("Задача 2:", interval_style(task2_score))
print("Задача 3:", interval_acc(task3_acc))
print("Задача 4:", interval_acc(task4_acc))
print("Задача 5:", interval_acc(task5_acc))


### Выводы (заполни после прогона)

- SFT переносит стиль, DPO закрепляет предпочтение → рост `P_simple` от задачи 1 к задаче 2.
- Reward Model на `good_bad` учится различать суть, потому что обе стороны пары в простом стиле — поверхностные признаки не помогают.
- DPO и SimPO решают одну задачу; SimPO обходится без reference-модели (экономия памяти), сравни итоговые accuracy и опиши, что это говорит о цене reference-модели.

### Чек-лист воспроизводимости
- [x] seed = 42 везде (python/numpy/torch + seed тренеров + `set_seed` перед генерацией)
- [x] генерация greedy (`do_sample=False`)
- [x] ячейки печатают каждую из пяти метрик
- [x] обучение целиком внутри ноутбука, адаптеры не подгружаются извне
- [ ] ноутбук запускается сверху вниз без ручных правок (проверь перезапуском!)
- [ ] коммить в репозиторий по ходу работы (после каждой задачи), без force-push